# CBRP Methodologies Analysis - R Notebook

This notebook contains analysis for the CBRP (Capacitated Bicycle Routing Problem) methodologies.


In [ ]:
install.packages("readxl")
install.packages("devtools")
devtools::install_github("b0rxa/scmamp")

In [42]:
# Load required libraries
library(scmamp)
library(readxl)

## Configuration

Set analysis parameters here.

In [87]:
# ============================================================================
# ANALYSIS CONFIGURATION
# ============================================================================

# Set to "LB" for Lower Bounds analysis or "UB" for Upper Bounds analysis
ANALYSIS_TYPE <- "UB"  # Options: "LB" or "UB"
# Read the lower bounds CSV
walk_lb_path <- "/home/carlos/Downloads/Walk - Upper Bounds.csv"
walk_lb_csv <- read.csv(walk_lb_path, stringsAsFactors = FALSE)

# Validate configuration
if (!ANALYSIS_TYPE %in% c("LB", "UB")) {
  stop("ANALYSIS_TYPE must be either 'LB' or 'UB'")
}

# Set analysis parameters based on type
if (ANALYSIS_TYPE == "LB") {
  BETTER_IS_HIGHER <- TRUE
  MISSING_VALUE <- 0
  RANK_SIGN <- -1  # Use rank(-x) so higher values get lower (better) ranks
  PLOTCD_DECREASING <- TRUE
  cat("=== ANALYSIS CONFIGURATION ===\n")
  cat("Type: LOWER BOUNDS\n")
  cat("Better values: HIGHER\n")
  cat("Missing data ('-') replaced with:", MISSING_VALUE, "\n")
  cat("Ranking: Higher values get better (lower) ranks\n\n")
} else {  # UB
  BETTER_IS_HIGHER <- FALSE
  MISSING_VALUE <- Inf
  RANK_SIGN <- 1   # Use rank(x) so lower values get lower (better) ranks
  PLOTCD_DECREASING <- FALSE
  cat("=== ANALYSIS CONFIGURATION ===\n")
  cat("Type: UPPER BOUNDS\n")
  cat("Better values: LOWER\n")
  cat("Missing data ('-') replaced with:", MISSING_VALUE, "\n")
  cat("Ranking: Lower values get better (lower) ranks\n\n")
}

=== ANALYSIS CONFIGURATION ===
Type: UPPER BOUNDS
Better values: LOWER
Missing data ('-') replaced with: Inf 
Ranking: Lower values get better (lower) ranks



## Data Loading

Load and prepare the data for analysis.


In [77]:
# Define path to the Excel file
excel_path <- "/home/carlos/Documentos/cbrp-methodologies/PhD - Deterministic Trail Models Results.xlsx"

# Get all sheet names
sheet_names <- excel_sheets(excel_path)

# Remove the "Compare-all" sheet if it exists
sheet_names <- sheet_names[!(sheet_names == "Compare-all" | grepl("Greedy", sheet_names, ignore.case = TRUE))]

# Clean and standardize method names
clean_names <- gsub("Trail ", "", sheet_names)
clean_names <- trimws(clean_names)

# Rename methods to standard naming convention
clean_names <- gsub("^Exp$", "Path-CBRP", clean_names)
clean_names <- gsub("^Exp Prep$", "Path-CBRP-Prep", clean_names)
clean_names <- gsub("^Exp Frac-Cut$", "Path-CBRP-Frac", clean_names)
clean_names <- gsub("^Exp Frac-Cut Prep$", "Path-CBRP-Frac-Prep", clean_names)
clean_names <- gsub("^MTZ$", "Path-CBRP-MTZ", clean_names)
clean_names <- gsub("^MTZ Prep$", "Path-CBRP-MTZ-Prep", clean_names)

# Read each remaining sheet into a list of dataframes
data_list <- lapply(sheet_names, function(sheet) {
  read_excel(excel_path, sheet = sheet)
})
names(data_list) <- clean_names

cat("\n=== EXCEL DATA LOADED ===\n")
cat("Methods found:", length(clean_names), "\n")
print(clean_names)

# ============================================================================
# BUILD LB AND UB DATAFRAMES FROM EXCEL
# ============================================================================

# Get all unique instances across all sheets
excel_instances <- sort(unique(unlist(lapply(data_list, function(df) df$Instance))))

# Initialize LB and UB dataframes
EXCEL_LB_df <- data.frame(row.names = excel_instances)
EXCEL_UB_df <- data.frame(row.names = excel_instances)

# Extract LB and UB from each method
for (method in names(data_list)) {
  df <- data_list[[method]]
  
  # Create named vectors for LB and UB
  lb_map <- setNames(excel_num(df$LB), df$Instance)
  ub_map <- setNames(excel_num(df$UB), df$Instance)
  
  # Add to dataframes
  EXCEL_LB_df[[method]] <- unname(lb_map[excel_instances])
  EXCEL_UB_df[[method]] <- unname(ub_map[excel_instances])
}

# Fill missing values
# For LB: missing means we don't have a lower bound, use 0
# For UB: missing means we don't have an upper bound, use Inf
EXCEL_LB_df[is.na(EXCEL_LB_df)] <- 0
EXCEL_UB_df[is.na(EXCEL_UB_df)] <- Inf

cat("\n=== EXCEL DATAFRAMES CREATED ===\n")
cat("EXCEL_LB_df dimensions:", nrow(EXCEL_LB_df), "instances x", ncol(EXCEL_LB_df), "methods\n")
cat("EXCEL_UB_df dimensions:", nrow(EXCEL_UB_df), "instances x", ncol(EXCEL_UB_df), "methods\n")
cat("\nMethods in dataframes:\n")
print(colnames(EXCEL_LB_df))
cat("\nFirst few rows of LB:\n")
print(head(EXCEL_LB_df, 3))
cat("\nFirst few rows of UB:\n")
print(head(EXCEL_UB_df, 3))



=== EXCEL DATA LOADED ===
Methods found: 6 
[1] "Path-CBRP"           "Path-CBRP-Frac"      "Path-CBRP-MTZ"      
[4] "Path-CBRP-Prep"      "Path-CBRP-Frac-Prep" "Path-CBRP-MTZ-Prep" 

=== EXCEL DATAFRAMES CREATED ===
EXCEL_LB_df dimensions: 39 instances x 6 methods
EXCEL_UB_df dimensions: 39 instances x 6 methods

Methods in dataframes:
[1] "Path-CBRP"           "Path-CBRP-Frac"      "Path-CBRP-MTZ"      
[4] "Path-CBRP-Prep"      "Path-CBRP-Frac-Prep" "Path-CBRP-MTZ-Prep" 

First few rows of LB:
                  Path-CBRP Path-CBRP-Frac Path-CBRP-MTZ Path-CBRP-Prep
alto-santo-1000-1        39             39            39             39
alto-santo-1000-2       240            240           240            240
alto-santo-1000-3       247            246           247            247
                  Path-CBRP-Frac-Prep Path-CBRP-MTZ-Prep
alto-santo-1000-1                  39                 39
alto-santo-1000-2                 240                240
alto-santo-1000-3                 247

In [88]:
# Pairwise win count function
# Compares methods pairwise and counts how many times each beats each other
pairwise_win_count <- function(df, win_type = c("highest", "lowest"), approaches_to_compare = NULL) {
  win_type <- match.arg(win_type)
  # Select only the approaches to be compared, or all if approaches_to_compare not provided
  if (!is.null(approaches_to_compare)) {
    approaches <- intersect(approaches_to_compare, colnames(df))
    sub_df <- df[, approaches, drop = FALSE]
  } else {
    approaches <- colnames(df)
    sub_df <- df
  }
  n_approaches <- length(approaches)
  win_matrix <- matrix(0, nrow = n_approaches, ncol = n_approaches,
                       dimnames = list(approaches, approaches))
  
  for (i in 1:nrow(sub_df)) {
    row_vals <- as.numeric(sub_df[i, ])
    for (a in 1:n_approaches) {
      for (b in 1:n_approaches) {
        # Do not compare same approach and only compare non-NA pairs
        if (a != b && !is.na(row_vals[a]) && !is.na(row_vals[b])) {
          if (win_type == "highest" && (row_vals[a] > row_vals[b])) {
            win_matrix[a, b] <- win_matrix[a, b] + 1
          }
          if (win_type == "lowest" && (row_vals[a] < row_vals[b])) {
            win_matrix[a, b] <- win_matrix[a, b] + 1
          }
        }
      }
    }
  }
  # Return the square matrix directly (not in melted data frame form)
  return(win_matrix)
}

In [89]:
# Helper: coerce Excel-imported numeric columns that may come as character
# Handles comma decimals ("12,34"), thousands separators ("1.234,56"), and missing values ("-")
excel_num <- function(x, use_missing_value = TRUE) {
  if (is.numeric(x)) return(x)
  x <- trimws(as.character(x))
  
  # Handle missing values represented as "-"
  if (use_missing_value && exists("MISSING_VALUE")) {
    missing_mask <- x == "-"
    x[missing_mask] <- as.character(MISSING_VALUE)
  }
  
  x[x == ""] <- NA

  has_dot <- grepl("\\.", x)
  has_comma <- grepl(",", x)

  y <- x
  both <- has_dot & has_comma
  y[both] <- gsub("\\.", "", y[both])      # remove thousands '.'
  y[has_comma] <- gsub(",", ".", y[has_comma]) # convert decimal ',' -> '.'

  result <- suppressWarnings(as.numeric(y))
  return(result)
}

In [90]:

# The first column is assumed to be the instance identifier
walk_lb_instances <- walk_lb_csv[[1]]
walk_lb_methods_orig <- names(walk_lb_csv)[-1] # All methods except the first column

# Rename methods changing '.' to '-'
walk_lb_methods <- gsub("\\.", "-", walk_lb_methods_orig)

cat("Methods found in CSV:\n")
print(walk_lb_methods)
cat("\nNumber of instances in CSV:", length(walk_lb_instances), "\n")

# Convert CSV data to the format expected by the analysis
# Each method from CSV becomes a named vector keyed by instance
walk_lb_best <- lapply(walk_lb_methods_orig, function(col) {
  vals <- (walk_lb_csv[[col]])
  names(vals) <- walk_lb_instances
  vals
})
names(walk_lb_best) <- walk_lb_methods

Methods found in CSV:
[1] "LR-RCSP-KN-HR-BM-Prep" "Walk-CBRP"             "Walk-CBRP-Prep"       
[4] "Walk-CBRP-MTZ"        

Number of instances in CSV: 39 


## Excel Bounds Analysis

Analysis using the ANALYSIS_TYPE flag to analyze either LB or UB from Excel data.

In [81]:
# Select the appropriate dataframe based on ANALYSIS_TYPE flag
if (ANALYSIS_TYPE == "LB") {
  EXCEL_BOUNDS_df <- EXCEL_LB_df
  cat("\n=== SELECTED: EXCEL LOWER BOUNDS ===\n")
} else {
  EXCEL_BOUNDS_df <- EXCEL_UB_df
  cat("\n=== SELECTED: EXCEL UPPER BOUNDS ===\n")
}

cat("Analysis Type:", ifelse(BETTER_IS_HIGHER, "Higher is better", "Lower is better"), "\n")
cat("Methods:", ncol(EXCEL_BOUNDS_df), "\n")
cat("Instances:", nrow(EXCEL_BOUNDS_df), "\n")
cat("\nMethods available:\n")
print(colnames(EXCEL_BOUNDS_df))
cat("\nFirst few rows:\n")
print(head(EXCEL_BOUNDS_df, 3))


=== SELECTED: EXCEL LOWER BOUNDS ===
Analysis Type: Higher is better 
Methods: 6 
Instances: 39 

Methods available:
[1] "Path-CBRP"           "Path-CBRP-Frac"      "Path-CBRP-MTZ"      
[4] "Path-CBRP-Prep"      "Path-CBRP-Frac-Prep" "Path-CBRP-MTZ-Prep" 

First few rows:
                  Path-CBRP Path-CBRP-Frac Path-CBRP-MTZ Path-CBRP-Prep
alto-santo-1000-1        39             39            39             39
alto-santo-1000-2       240            240           240            240
alto-santo-1000-3       247            246           247            247
                  Path-CBRP-Frac-Prep Path-CBRP-MTZ-Prep
alto-santo-1000-1                  39                 39
alto-santo-1000-2                 240                240
alto-santo-1000-3                 247                247


In [82]:
# Statistical tests for EXCEL methods
# Iman-Davenport test and Nemenyi post-hoc test

if (ncol(EXCEL_BOUNDS_df) >= 2) {
  cat("\n=== STATISTICAL TESTS FOR EXCEL", ANALYSIS_TYPE, "===\n")
  cat("Analysis Type:", ifelse(BETTER_IS_HIGHER, "Higher is better", "Lower is better"), "\n")
  
  # Critical value calculation
  k_EXCEL <- ncol(EXCEL_BOUNDS_df)
  N_EXCEL <- nrow(EXCEL_BOUNDS_df)
  CriticalValue_EXCEL <- qf(0.95, k_EXCEL - 1, (k_EXCEL - 1) * (N_EXCEL - 1))
  cat("Critical Value (F-distribution, alpha=0.05):", CriticalValue_EXCEL, "\n\n")
  
  # Iman-Davenport Test
  cat("--- Iman-Davenport Test ---\n")
  res_id_excel_bounds <- imanDavenportTest(EXCEL_BOUNDS_df)
  print(res_id_excel_bounds[['statistic']])
  cat("\nInterpretation: If statistic >", CriticalValue_EXCEL, "there are significant differences\n")
  
  # Nemenyi Post-hoc Test
  cat("\n--- Nemenyi Post-hoc Test ---\n")
  res_nemenyi_excel_bounds <- nemenyiTest(EXCEL_BOUNDS_df)
  print(res_nemenyi_excel_bounds[['statistic']])
  cat("\nCritical Difference:", res_nemenyi_excel_bounds[['statistic']], "\n")
  cat("Methods with rank differences > CD are significantly different\n")
  
  # Compute and display average ranks
  cat("\n--- Average Ranks (lower rank = better, rank 1 is best) ---\n")
  # Use RANK_SIGN to determine ranking direction based on analysis type
  ranks_matrix_excel <- apply(EXCEL_BOUNDS_df, 1, function(x) rank(RANK_SIGN * x, ties.method = "average"))
  if (!is.matrix(ranks_matrix_excel)) ranks_matrix_excel <- matrix(ranks_matrix_excel, nrow = k_EXCEL)
  ranks_matrix_excel <- t(ranks_matrix_excel)
  avg_ranks_excel <- colMeans(ranks_matrix_excel)
  avg_ranks_sorted_excel <- sort(avg_ranks_excel) # show best (lowest rank) first
  print(avg_ranks_sorted_excel)
  
} else {
  cat("\nNeed at least 2 methods for statistical comparison\n")
}


=== STATISTICAL TESTS FOR EXCEL LB ===
Analysis Type: Higher is better 
Critical Value (F-distribution, alpha=0.05): 2.261638 

--- Iman-Davenport Test ---
Corrected Friedman's chi-squared 
                        7.343958 

Interpretation: If statistic > 2.261638 there are significant differences

--- Nemenyi Post-hoc Test ---
Critical difference 
           1.217646 

Critical Difference: 1.217646 
Methods with rank differences > CD are significantly different

--- Average Ranks (lower rank = better, rank 1 is best) ---
     Path-CBRP-Prep           Path-CBRP  Path-CBRP-MTZ-Prep       Path-CBRP-MTZ 
           2.884615            2.923077            3.025641            3.256410 
Path-CBRP-Frac-Prep      Path-CBRP-Frac 
           4.371795            4.538462 


In [83]:
# Generate Critical Difference (CD) plot for EXCEL methods

if (ncol(EXCEL_BOUNDS_df) >= 2) {
  cat("\n=== GENERATING CD PLOT FOR EXCEL METHODS ===\n")
  
  N_EXCEL <- nrow(EXCEL_BOUNDS_df)
  k_EXCEL <- ncol(EXCEL_BOUNDS_df)
  
  # Compute ranks using configured ranking direction
  ranks_matrix_excel <- apply(EXCEL_BOUNDS_df, 1, function(x) rank(RANK_SIGN * x, ties.method = "average"))
  if (!is.matrix(ranks_matrix_excel)) ranks_matrix_excel <- matrix(ranks_matrix_excel, nrow = k_EXCEL)
  ranks_matrix_excel <- t(ranks_matrix_excel) # N x k
  
  R_j_EXCEL <- colMeans(ranks_matrix_excel)
  
  # Create labels with average ranks
  approach_labels_excel <- colnames(EXCEL_BOUNDS_df)
  for (j in 1:k_EXCEL) {
    approach_labels_excel[j] <- sprintf("%s\n(%.2f)", colnames(EXCEL_BOUNDS_df)[j], R_j_EXCEL[j])
  }
  
  EXCEL_BOUNDS_df_plot <- EXCEL_BOUNDS_df
  colnames(EXCEL_BOUNDS_df_plot) <- approach_labels_excel
  
  # Generate PDF plot using configured direction
  pdf_filename <- paste0("cd_excel_", tolower(ANALYSIS_TYPE), ".pdf")
  pdf(pdf_filename, width=12, height=6)
  plotCD(EXCEL_BOUNDS_df_plot, cex=1, decreasing=PLOTCD_DECREASING)
  dev.off()
  
  cat("CD plot saved to:", pdf_filename, "\n")
} else {
  cat("\nNeed at least 2 methods to generate CD plot\n")
}


=== GENERATING CD PLOT FOR EXCEL METHODS ===
CD plot saved to: cd_excel_lb.pdf 


In [84]:
# Pairwise win counts for EXCEL methods
# Analysis focuses on statistically equivalent methods with the best rank

if (ncol(EXCEL_BOUNDS_df) >= 2 && exists("pairwise_win_count")) {
  cat("\n=== PAIRWISE WIN ANALYSIS FOR EXCEL METHODS ===\n")
  
  # Get average ranks and critical difference from previous analysis
  if (exists("res_nemenyi_excel_bounds") && exists("avg_ranks_excel")) {
    CD <- res_nemenyi_excel_bounds[['statistic']]
    
    # Find the best (lowest) rank
    best_rank <- min(avg_ranks_excel)
    best_method <- names(which.min(avg_ranks_excel))
    
    cat("\nBest method (lowest rank):", best_method, "with rank", round(best_rank, 2), "\n")
    cat("Critical Difference (CD):", round(CD, 4), "\n")
    
    # Find all methods statistically equivalent to the best
    # Methods are equivalent if |rank_i - rank_best| <= CD
    equivalent_to_best <- names(avg_ranks_excel[abs(avg_ranks_excel - best_rank) <= CD])
    
    cat("\nMethods statistically equivalent to the best (within CD):\n")
    for (method in equivalent_to_best) {
      cat(sprintf("  - %s (rank: %.2f, diff: %.2f)\n", 
                  method, avg_ranks_excel[method], abs(avg_ranks_excel[method] - best_rank)))
    }
    
    # Perform pairwise comparison only among statistically equivalent methods
    cat("\n--- Pairwise Win Analysis (Equivalent Methods Only) ---\n")
    cat("Comparing only the", length(equivalent_to_best), "statistically equivalent methods\n")
    cat("Win type:", ifelse(BETTER_IS_HIGHER, "Higher is better", "Lower is better"), "\n\n")
    
    win_type_param <- ifelse(BETTER_IS_HIGHER, "highest", "lowest")
    EXCEL_pairwise_win_matrix <- pairwise_win_count(EXCEL_BOUNDS_df, 
                                                    win_type = win_type_param,
                                                    approaches_to_compare = equivalent_to_best)
    cat("Pairwise win counts:\n")
    print(EXCEL_pairwise_win_matrix)
    
    # Replace diagonal with hyphens for display
    EXCEL_pairwise_win_matrix_disp <- EXCEL_pairwise_win_matrix
    diag(EXCEL_pairwise_win_matrix_disp) <- "-"
    
    cat("\nPairwise win counts [with '-' on diagonal]:\n")
    print(EXCEL_pairwise_win_matrix_disp)
    
    # Generate LaTeX table
    library(xtable)
    EXCEL_pairwise_win_matrix_disp_df <- as.data.frame.matrix(EXCEL_pairwise_win_matrix_disp)
    EXCEL_pairwise_win_matrix_disp_df[] <- lapply(EXCEL_pairwise_win_matrix_disp_df, as.character)
    
    caption_text <- paste0("Pairwise win counts for statistically equivalent EXCEL ", ANALYSIS_TYPE, " approaches.")
    cat("\n--- LaTeX Table (Statistically Equivalent Methods) ---\n")
    print(xtable(EXCEL_pairwise_win_matrix_disp_df, 
                 caption = caption_text, 
                 label = "tab:excel_pairwise_wins_equiv",
                 align = c("l", rep("c", ncol(EXCEL_pairwise_win_matrix_disp_df)))),
          include.rownames=TRUE, sanitize.text.function=identity)
    
  } else {
    cat("\nPlease run the statistical tests cell first (Cell 12)\n")
  }
  
  # Also show full comparison for reference
  cat("\n\n=== FULL PAIRWISE COMPARISON (ALL EXCEL METHODS) ===\n")
  win_type_param <- ifelse(BETTER_IS_HIGHER, "highest", "lowest")
  EXCEL_pairwise_win_matrix_full <- pairwise_win_count(EXCEL_BOUNDS_df, win_type = win_type_param)
  cat("Pairwise win counts (all methods):\n")
  print(EXCEL_pairwise_win_matrix_full)
  
} else if (!exists("pairwise_win_count")) {
  cat("\nPlease run the cell that defines pairwise_win_count function first\n")
} else {
  cat("\nNeed at least 2 methods for pairwise comparison\n")
}


=== PAIRWISE WIN ANALYSIS FOR EXCEL METHODS ===

Best method (lowest rank): Path-CBRP-Prep with rank 2.88 
Critical Difference (CD): 1.2176 

Methods statistically equivalent to the best (within CD):
  - Path-CBRP (rank: 2.92, diff: 0.04)
  - Path-CBRP-MTZ (rank: 3.26, diff: 0.37)
  - Path-CBRP-Prep (rank: 2.88, diff: 0.00)
  - Path-CBRP-MTZ-Prep (rank: 3.03, diff: 0.14)

--- Pairwise Win Analysis (Equivalent Methods Only) ---
Comparing only the 4 statistically equivalent methods
Win type: Higher is better 

Pairwise win counts:
                   Path-CBRP Path-CBRP-MTZ Path-CBRP-Prep Path-CBRP-MTZ-Prep
Path-CBRP                  0            11              8                 12
Path-CBRP-MTZ              6             0              3                  6
Path-CBRP-Prep             9            10              0                  8
Path-CBRP-MTZ-Prep         7            11              7                  0

Pairwise win counts [with '-' on diagonal]:
                   Path-CBRP Path-

## CSV Lower Bounds Analysis

Analysis of lower bounds from CSV file - comparing different approaches.

In [65]:
# Create a data frame specifically for CSV methods comparison
# This allows focused statistical analysis on the approaches from CSV

# Build CSV-only dataframe
CSV_BOUNDS_df <- data.frame(row.names = walk_lb_instances)

for (method in walk_lb_methods) {
  bound_vector <- walk_lb_best[[method]]
  CSV_BOUNDS_df[[method]] <- unname(bound_vector[walk_lb_instances])
}

# Remove any rows with all NAs
CSV_BOUNDS_df <- CSV_BOUNDS_df[rowSums(!is.na(CSV_BOUNDS_df)) > 0, , drop = FALSE]

# Fill remaining NAs with configured MISSING_VALUE
CSV_BOUNDS_df[is.na(CSV_BOUNDS_df)] <- MISSING_VALUE

cat("\n=== CSV", ANALYSIS_TYPE, "DATA SUMMARY ===\n")
cat("Analysis Type:", ifelse(ANALYSIS_TYPE == "LB", "Lower Bounds (higher is better)", "Upper Bounds (lower is better)"), "\n")
cat("Methods from CSV:", ncol(CSV_BOUNDS_df), "\n")
cat("Instances:", nrow(CSV_BOUNDS_df), "\n")
cat("\nMethods available:\n")
print(colnames(CSV_BOUNDS_df))
cat("\nFirst few rows:\n")
print(head(CSV_BOUNDS_df, 3))


=== CSV LB DATA SUMMARY ===
Analysis Type: Lower Bounds (higher is better) 
Methods from CSV: 6 
Instances: 39 

Methods available:
[1] "Greedy-Heuristic"      "LR-RCSP-KN-HR-BM"      "LR-RCSP-KN-HR-BM-Prep"
[4] "Walk-CBRP"             "Walk-CBRP-Prep"        "Walk-CBRP-MTZ"        

First few rows:
                  Greedy-Heuristic LR-RCSP-KN-HR-BM LR-RCSP-KN-HR-BM-Prep
alto-santo-1000-1               39               39                    39
alto-santo-1000-2              238              238                   238
alto-santo-1000-3              245              245                   245
                  Walk-CBRP Walk-CBRP-Prep Walk-CBRP-MTZ
alto-santo-1000-1        39             39            39
alto-santo-1000-2        49            238           237
alto-santo-1000-3        50            246           240


In [66]:
# Statistical tests for CSV methods
# Iman-Davenport test and Nemenyi post-hoc test

if (ncol(CSV_BOUNDS_df) >= 2) {
  cat("\n=== STATISTICAL TESTS FOR CSV", ANALYSIS_TYPE, "===\n")
  cat("Analysis Type:", ifelse(BETTER_IS_HIGHER, "Higher is better", "Lower is better"), "\n")
  
  # Critical value calculation
  k_CSV <- ncol(CSV_BOUNDS_df)
  N_CSV <- nrow(CSV_BOUNDS_df)
  CriticalValue_CSV <- qf(0.95, k_CSV - 1, (k_CSV - 1) * (N_CSV - 1))
  cat("Critical Value (F-distribution, alpha=0.05):", CriticalValue_CSV, "\n\n")
  
  # Iman-Davenport Test
  cat("--- Iman-Davenport Test ---\n")
  res_id_csv_bounds <- imanDavenportTest(CSV_BOUNDS_df)
  print(res_id_csv_bounds[['statistic']])
  cat("\nInterpretation: If statistic >", CriticalValue_CSV, "there are significant differences\n")
  
  # Nemenyi Post-hoc Test
  cat("\n--- Nemenyi Post-hoc Test ---\n")
  res_nemenyi_csv_bounds <- nemenyiTest(CSV_BOUNDS_df)
  print(res_nemenyi_csv_bounds[['statistic']])
  cat("\nCritical Difference:", res_nemenyi_csv_bounds[['statistic']], "\n")
  cat("Methods with rank differences > CD are significantly different\n")
  
  # Compute and display average ranks
  cat("\n--- Average Ranks (lower rank = better, rank 1 is best) ---\n")
  # Use RANK_SIGN to determine ranking direction based on analysis type
  ranks_matrix_csv <- apply(CSV_BOUNDS_df, 1, function(x) rank(RANK_SIGN * x, ties.method = "average"))
  if (!is.matrix(ranks_matrix_csv)) ranks_matrix_csv <- matrix(ranks_matrix_csv, nrow = k_CSV)
  ranks_matrix_csv <- t(ranks_matrix_csv)
  avg_ranks_csv <- colMeans(ranks_matrix_csv)
  avg_ranks_sorted <- sort(avg_ranks_csv) # show best (lowest rank) first
  print(avg_ranks_sorted)
  
} else {
  cat("\nNeed at least 2 methods for statistical comparison\n")
}


=== STATISTICAL TESTS FOR CSV LB ===
Analysis Type: Higher is better 
Critical Value (F-distribution, alpha=0.05): 2.261638 

--- Iman-Davenport Test ---
Corrected Friedman's chi-squared 
                        29.16235 

Interpretation: If statistic > 2.261638 there are significant differences

--- Nemenyi Post-hoc Test ---
Critical difference 
           1.217646 

Critical Difference: 1.217646 
Methods with rank differences > CD are significantly different

--- Average Ranks (lower rank = better, rank 1 is best) ---
     Greedy-Heuristic      LR-RCSP-KN-HR-BM LR-RCSP-KN-HR-BM-Prep 
             2.487179              2.487179              2.641026 
       Walk-CBRP-Prep             Walk-CBRP         Walk-CBRP-MTZ 
             3.538462              4.307692              5.538462 


In [67]:
# Generate Critical Difference (CD) plot for CSV methods

if (ncol(CSV_BOUNDS_df) >= 2) {
  cat("\n=== GENERATING CD PLOT FOR CSV METHODS ===\n")
  
  N_CSV <- nrow(CSV_BOUNDS_df)
  k_CSV <- ncol(CSV_BOUNDS_df)
  
  # Compute ranks using configured ranking direction
  ranks_matrix_csv <- apply(CSV_BOUNDS_df, 1, function(x) rank(RANK_SIGN * x, ties.method = "average"))
  if (!is.matrix(ranks_matrix_csv)) ranks_matrix_csv <- matrix(ranks_matrix_csv, nrow = k_CSV)
  ranks_matrix_csv <- t(ranks_matrix_csv) # N x k
  
  R_j_CSV <- colMeans(ranks_matrix_csv)
  
  # Create labels with average ranks
  approach_labels_csv <- colnames(CSV_BOUNDS_df)
  for (j in 1:k_CSV) {
    approach_labels_csv[j] <- sprintf("%s\n(%.2f)", colnames(CSV_BOUNDS_df)[j], R_j_CSV[j])
  }
  
  CSV_BOUNDS_df_plot <- CSV_BOUNDS_df
  colnames(CSV_BOUNDS_df_plot) <- approach_labels_csv
  
  # Generate PDF plot using configured direction
  pdf_filename <- paste0("cd_csv_walk_", tolower(ANALYSIS_TYPE), ".pdf")
  pdf(pdf_filename, width=12, height=6)
  plotCD(CSV_BOUNDS_df_plot, cex=1, decreasing=PLOTCD_DECREASING)
  dev.off()
  
  cat("CD plot saved to:", pdf_filename, "\n")
} else {
  cat("\nNeed at least 2 methods to generate CD plot\n")
}


=== GENERATING CD PLOT FOR CSV METHODS ===
CD plot saved to: cd_csv_walk_lb.pdf 


In [58]:
# Pairwise win counts for CSV methods
# Analysis focuses on statistically equivalent methods with the best rank

if (ncol(CSV_BOUNDS_df) >= 2 && exists("pairwise_win_count")) {
  cat("\n=== PAIRWISE WIN ANALYSIS FOR CSV METHODS ===\n")
  
  # Get average ranks and critical difference from previous analysis
  if (exists("res_nemenyi_csv_bounds") && exists("avg_ranks_csv")) {
    CD <- res_nemenyi_csv_bounds[['statistic']]
    
    # Find the best (lowest) rank
    best_rank <- min(avg_ranks_csv)
    best_method <- names(which.min(avg_ranks_csv))
    
    cat("\nBest method (lowest rank):", best_method, "with rank", round(best_rank, 2), "\n")
    cat("Critical Difference (CD):", round(CD, 4), "\n")
    
    # Find all methods statistically equivalent to the best
    # Methods are equivalent if |rank_i - rank_best| <= CD
    equivalent_to_best <- names(avg_ranks_csv[abs(avg_ranks_csv - best_rank) <= CD])
    
    cat("\nMethods statistically equivalent to the best (within CD):\n")
    for (method in equivalent_to_best) {
      cat(sprintf("  - %s (rank: %.2f, diff: %.2f)\n", 
                  method, avg_ranks_csv[method], abs(avg_ranks_csv[method] - best_rank)))
    }
    
    # Perform pairwise comparison only among statistically equivalent methods
    cat("\n--- Pairwise Win Analysis (Equivalent Methods Only) ---\n")
    cat("Comparing only the", length(equivalent_to_best), "statistically equivalent methods\n")
    cat("Win type:", ifelse(BETTER_IS_HIGHER, "Higher is better", "Lower is better"), "\n\n")
    
    win_type_param <- ifelse(BETTER_IS_HIGHER, "highest", "lowest")
    CSV_pairwise_win_matrix <- pairwise_win_count(CSV_BOUNDS_df, 
                                                    win_type = win_type_param,
                                                    approaches_to_compare = equivalent_to_best)
    cat("Pairwise win counts:\n")
    print(CSV_pairwise_win_matrix)
    
    # Replace diagonal with hyphens for display
    CSV_pairwise_win_matrix_disp <- CSV_pairwise_win_matrix
    diag(CSV_pairwise_win_matrix_disp) <- "-"
    
    cat("\nPairwise win counts [with '-' on diagonal]:\n")
    print(CSV_pairwise_win_matrix_disp)
    
    # Generate LaTeX table
    library(xtable)
    CSV_pairwise_win_matrix_disp_df <- as.data.frame.matrix(CSV_pairwise_win_matrix_disp)
    CSV_pairwise_win_matrix_disp_df[] <- lapply(CSV_pairwise_win_matrix_disp_df, as.character)
    
    caption_text <- paste0("Pairwise win counts for statistically equivalent CSV ", ANALYSIS_TYPE, " approaches.")
    cat("\n--- LaTeX Table (Statistically Equivalent Methods) ---\n")
    print(xtable(CSV_pairwise_win_matrix_disp_df, 
                 caption = caption_text, 
                 label = "tab:csv_pairwise_wins_equiv",
                 align = c("l", rep("c", ncol(CSV_pairwise_win_matrix_disp_df)))),
          include.rownames=TRUE, sanitize.text.function=identity)
    
  } else {
    cat("\nPlease run the statistical tests cell first (Cell 13)\n")
  }
  
  # Also show full comparison for reference
  cat("\n\n=== FULL PAIRWISE COMPARISON (ALL CSV METHODS) ===\n")
  win_type_param <- ifelse(BETTER_IS_HIGHER, "highest", "lowest")
  CSV_pairwise_win_matrix_full <- pairwise_win_count(CSV_BOUNDS_df, win_type = win_type_param)
  cat("Pairwise win counts (all methods):\n")
  print(CSV_pairwise_win_matrix_full)
  
} else if (!exists("pairwise_win_count")) {
  cat("\nPlease run the cell that defines pairwise_win_count function first\n")
} else {
  cat("\nNeed at least 2 methods for pairwise comparison\n")
}


=== PAIRWISE WIN ANALYSIS FOR CSV METHODS ===

Best method (lowest rank): Walk-CBRP-Prep with rank 1.51 
Critical Difference (CD): 0.7594 

Methods statistically equivalent to the best (within CD):
  - Walk-CBRP (rank: 2.09, diff: 0.58)
  - Walk-CBRP-Prep (rank: 1.51, diff: 0.00)

--- Pairwise Win Analysis (Equivalent Methods Only) ---
Comparing only the 2 statistically equivalent methods
Win type: Lower is better 

Pairwise win counts:
               Walk-CBRP Walk-CBRP-Prep
Walk-CBRP              0              6
Walk-CBRP-Prep        18              0

Pairwise win counts [with '-' on diagonal]:
               Walk-CBRP Walk-CBRP-Prep
Walk-CBRP      "-"       "6"           
Walk-CBRP-Prep "18"      "-"           

--- LaTeX Table (Statistically Equivalent Methods) ---
% latex table generated in R 4.3.3 by xtable 1.8-4 package
% Sun Feb  1 17:07:24 2026
\begin{table}[ht]
\centering
\begin{tabular}{lcc}
  \hline
 & Walk-CBRP & Walk-CBRP-Prep \\ 
  \hline
Walk-CBRP & - & 6 \\ 
  Walk-

## Exploratory Data Analysis

View and summarize the loaded data.


In [30]:
# IMPORTANT: sheets may have different row orders/filters.
# Build comparison data frames by aligning rows using the 'Instance' key.

# Get all instances from Excel sheets
instances_excel <- sort(unique(unlist(lapply(data_list, function(df) df$Instance))))

# Get instances from CSV
instances_csv <- walk_lb_instances

# Combine all instances (union of both sources)
instances_all <- sort(unique(c(instances_excel, instances_csv)))

cat("\n=== Data Integration Summary ===\n")
cat("Instances from Excel:", length(instances_excel), "\n")
cat("Instances from CSV:", length(instances_csv), "\n")
cat("Total unique instances:", length(instances_all), "\n\n")

# Initialize dataframes
LB_df <- data.frame(row.names = instances_all)
UB_df <- data.frame(row.names = instances_all)

# Add data from Excel sheets (existing code)
for (method in names(data_list)) {
  df <- data_list[[method]]
  # named vectors keyed by Instance
  lb_map <- setNames(excel_num(df$LB), df$Instance)
  ub_map <- setNames(excel_num(df$UB), df$Instance)

  LB_df[[method]] <- unname(lb_map[instances_all])
  UB_df[[method]] <- unname(ub_map[instances_all])
}

# Add data from CSV (lower bounds only)
for (method in walk_lb_methods) {
  lb_vector <- walk_lb_best[[method]]
  # Map CSV values to all instances
  LB_df[[method]] <- unname(lb_vector[instances_all])
}

# Fill missing instances (when a sheet doesn't contain some Instance rows)
LB_df[is.na(LB_df)] <- 0
UB_df[is.na(UB_df)] <- Inf

cat("LB_df dimensions:", nrow(LB_df), "instances x", ncol(LB_df), "methods\n")
cat("UB_df dimensions:", nrow(UB_df), "instances x", ncol(UB_df), "methods\n")
cat("\nMethods in LB_df:\n")
print(colnames(LB_df))
cat("\nMethods in UB_df:\n")
print(colnames(UB_df))



=== Data Integration Summary ===
Instances from Excel: 39 
Instances from CSV: 39 
Total unique instances: 39 

LB_df dimensions: 39 instances x 8 methods
UB_df dimensions: 39 instances x 5 methods

Methods in LB_df:
[1] "Walk-CBRP"             "Walk-CBRP-Frac"        "Walk-CBRP-Frac-Prep"  
[4] "Walk-CBRP-Prep"        "Walk-CBRP-MTZ"         "Greedy-Heuristic"     
[7] "LR-RCSP-KN-HR-BM"      "LR-RCSP-KN-HR-BM-Prep"

Methods in UB_df:
[1] "Walk-CBRP"           "Walk-CBRP-Frac"      "Walk-CBRP-Frac-Prep"
[4] "Walk-CBRP-Prep"      "Walk-CBRP-MTZ"      


In [ ]:
# Compute the CriticalValue for LB_df and UB_df using the qf function.
# Normally, for the Iman-Davenport test, the critical value is based on the F-distribution.

# For LB_df
k_LB <- ncol(LB_df)
N_LB <- nrow(LB_df)
CriticalValue_LB <- qf(0.95, k_LB - 1, (k_LB - 1) * (N_LB - 1))
print(paste("Critical Value for LB_df:", CriticalValue_LB))

# For UB_df
k_UB <- ncol(UB_df)
N_UB <- nrow(UB_df)
CriticalValue_UB <- qf(0.95, k_UB - 1, (k_UB - 1) * (N_UB - 1))
print(paste("Critical Value for UB_df:", CriticalValue_UB))


[1] "Critical Value for LB_df: 2.43116424264913"
[1] "Critical Value for UB_df: 2.43116424264913"


In [ ]:
print("Lower Bound")
res_id_lb = imanDavenportTest(LB_df)
print(res_id_lb[['statistic']])

res_nemenyi_lb = nemenyiTest(LB_df)
print(res_nemenyi_lb[['statistic']])

[1] "Lower Bound"
Corrected Friedman's chi-squared 
                        9.527342 
Critical difference 
          0.9861445 


In [ ]:
print("Upper Bound")
res_id_ub = imanDavenportTest(UB_df)
print(res_id_ub[['statistic']])

res_nemenyi_ub = nemenyiTest(UB_df)
print(res_nemenyi_ub[['statistic']])

[1] "Upper Bound"
Corrected Friedman's chi-squared 
                        17.95701 
Critical difference 
          0.9861445 


## Visualization

Create plots and visualizations of the data.


In [ ]:
# ==== Plot for LB_df ====
N_LB <- nrow(LB_df)
k_LB <- ncol(LB_df)

ranks_matrix_LB <- apply(LB_df, 1, rank)
if (!is.matrix(ranks_matrix_LB)) ranks_matrix_LB <- matrix(ranks_matrix_LB, nrow = k_LB)
ranks_matrix_LB <- t(ranks_matrix_LB) # N x k

R_j_LB <- colMeans(ranks_matrix_LB)

approach_labels_LB <- colnames(LB_df)
for (j in 1:k_LB) {
  approach_labels_LB[j] <- sprintf("%s\n(%.2f)", colnames(LB_df)[j], R_j_LB[j])
}
LB_df_plot <- LB_df
colnames(LB_df_plot) <- approach_labels_LB

pdf("cd_walk_models_lb.pdf", width=12, height=6)
plotCD(LB_df_plot, cex=1, decreasing=FALSE)
dev.off()

# ==== Plot for UB_df ====
N_UB <- nrow(UB_df)
k_UB <- ncol(UB_df)

ranks_matrix_UB <- apply(UB_df, 1, rank)
if (!is.matrix(ranks_matrix_UB)) ranks_matrix_UB <- matrix(ranks_matrix_UB, nrow = k_UB)
ranks_matrix_UB <- t(ranks_matrix_UB) # N x k

R_j_UB <- colMeans(ranks_matrix_UB)

approach_labels_UB <- colnames(UB_df)
for (j in 1:k_UB) {
  approach_labels_UB[j] <- sprintf("%s\n(%.2f)", colnames(UB_df)[j], R_j_UB[j])
}
UB_df_plot <- UB_df
colnames(UB_df_plot) <- approach_labels_UB

pdf("cd_walk_models_ub.pdf", width=12, height=6)
plotCD(UB_df_plot, cex=1, decreasing=FALSE)
dev.off()


agg_record_297074224 
                   2

agg_record_297074224 
                   2

In [ ]:
pairwise_win_count <- function(df, win_type = c("highest", "lowest"), approaches_to_compare = NULL) {
  win_type <- match.arg(win_type)
  # Select only the approaches to be compared, or all if approaches_to_compare not provided
  if (!is.null(approaches_to_compare)) {
    approaches <- intersect(approaches_to_compare, colnames(df))
    sub_df <- df[, approaches, drop = FALSE]
  } else {
    approaches <- colnames(df)
    sub_df <- df
  }
  n_approaches <- length(approaches)
  win_matrix <- matrix(0, nrow = n_approaches, ncol = n_approaches,
                       dimnames = list(approaches, approaches))
  
  for (i in 1:nrow(sub_df)) {
    row_vals <- as.numeric(sub_df[i, ])
    for (a in 1:n_approaches) {
      for (b in 1:n_approaches) {
        # Do not compare same approach and only compare non-NA pairs
        if (a != b && !is.na(row_vals[a]) && !is.na(row_vals[b])) {
          if (win_type == "highest" && (row_vals[a] > row_vals[b])) {
            win_matrix[a, b] <- win_matrix[a, b] + 1
          }
          if (win_type == "lowest" && (row_vals[a] < row_vals[b])) {
            win_matrix[a, b] <- win_matrix[a, b] + 1
          }
        }
      }
    }
  }
  # Return the square matrix directly (not in melted data frame form)
  return(win_matrix)
}


In [ ]:
# Pairwise win counts for Lower Bounds (higher is better)
# Use all methods available in LB_df
cat("\n=== PAIRWISE WIN ANALYSIS FOR LOWER BOUNDS ===\n")
cat("Comparing all", ncol(LB_df), "methods in LB_df\n\n")

LB_pairwise_win_matrix <- pairwise_win_count(LB_df, win_type = "highest")
print("Pairwise win counts for LB (Higher is better):")
print(LB_pairwise_win_matrix)

# Replace zeros on the diagonal with hyphen "-"
LB_pairwise_win_matrix_disp <- LB_pairwise_win_matrix
diag(LB_pairwise_win_matrix_disp) <- "-"
print("\nPairwise win counts for LB (Higher is better) [with '-' on diagonal]:")
print(LB_pairwise_win_matrix_disp)

# Generate LaTeX table
library(xtable)
# Convert matrix with hyphens to data.frame for xtable compatibility
LB_pairwise_win_matrix_disp_df <- as.data.frame.matrix(LB_pairwise_win_matrix_disp)
# xtable will attempt to coerce numeric columns; force all to character to preserve "-"
LB_pairwise_win_matrix_disp_df[] <- lapply(LB_pairwise_win_matrix_disp_df, as.character)
cat("\nWin counts Among Model LBs (LaTeX table):\n")
print(xtable(LB_pairwise_win_matrix_disp_df, 
             caption = "Win counts Among Model LBs.", 
             align = c("l", rep("c", ncol(LB_pairwise_win_matrix_disp_df)))),
      include.rownames=TRUE, sanitize.text.function=identity)

[1] "Pairwise win counts for LB (Higher is better):"
                    Walk-CBRP Walk-CBRP-Frac Walk-CBRP-Frac-Prep Walk-CBRP-Prep
Walk-CBRP                   0              0                  14             14
Walk-CBRP-Frac              0              0                  14             14
Walk-CBRP-Frac-Prep        17             17                   0              0
Walk-CBRP-Prep             17             17                   0              0
Walk-CBRP-MTZ              10             10                   3              3
                    Walk-CBRP-MTZ
Walk-CBRP                      27
Walk-CBRP-Frac                 27
Walk-CBRP-Frac-Prep            34
Walk-CBRP-Prep                 34
Walk-CBRP-MTZ                   0
[1] "Pairwise win counts for LB (Higher is better) [with '-' on diagonal]:"
                    Walk-CBRP Walk-CBRP-Frac Walk-CBRP-Frac-Prep Walk-CBRP-Prep
Walk-CBRP           "-"       "0"            "14"                "14"          
Walk-CBRP-Frac      "0"    

Win counts Among Model LBs
% latex table generated in R 4.5.2 by xtable 1.8-4 package
% Sat Jan 10 18:22:02 2026
\begin{table}[ht]
\centering
\begin{tabular}{lccccc}
  \hline
 & Walk-CBRP & Walk-CBRP-Frac & Walk-CBRP-Frac-Prep & Walk-CBRP-Prep & Walk-CBRP-MTZ \\ 
  \hline
Walk-CBRP & - & 0 & 14 & 14 & 27 \\ 
  Walk-CBRP-Frac & 0 & - & 14 & 14 & 27 \\ 
  Walk-CBRP-Frac-Prep & 17 & 17 & - & 0 & 34 \\ 
  Walk-CBRP-Prep & 17 & 17 & 0 & - & 34 \\ 
  Walk-CBRP-MTZ & 10 & 10 & 3 & 3 & - \\ 
   \hline
\end{tabular}
\caption{Win counts Among Model LBs.} 
\end{table}


In [ ]:
# Pairwise win counts for Upper Bounds (lower is better)
# Use all methods available in UB_df
cat("\n=== PAIRWISE WIN ANALYSIS FOR UPPER BOUNDS ===\n")
cat("Comparing all", ncol(UB_df), "methods in UB_df\n\n")

UB_pairwise_win_matrix <- pairwise_win_count(UB_df, win_type = "lowest")
print("Pairwise win counts for UB (Lower is better):")
print(UB_pairwise_win_matrix)

# Replace zeros on the diagonal with hyphen "-"
UB_pairwise_win_matrix_disp <- UB_pairwise_win_matrix
diag(UB_pairwise_win_matrix_disp) <- "-"
print("\nPairwise win counts for UB (Lower is better) [with '-' on diagonal]:")
print(UB_pairwise_win_matrix_disp)

# Generate LaTeX table
library(xtable)
# Convert matrix with hyphens to data.frame for xtable compatibility
UB_pairwise_win_matrix_disp_df <- as.data.frame.matrix(UB_pairwise_win_matrix_disp)
# xtable will attempt to coerce numeric columns; force all to character to preserve "-"
UB_pairwise_win_matrix_disp_df[] <- lapply(UB_pairwise_win_matrix_disp_df, as.character)
cat("\nWin counts Among Model UBs (LaTeX table):\n")
print(xtable(UB_pairwise_win_matrix_disp_df, 
             caption = "Win counts Among Model UBs.", 
             align = c("l", rep("c", ncol(UB_pairwise_win_matrix_disp_df)))),
      include.rownames=TRUE, sanitize.text.function=identity)

[1] "Pairwise win counts for UB (Lower is better):"
                   Path-CBRP-MTZ Path-CBRP-MTZ-Prep Path-CBRP-Prep
Path-CBRP-MTZ                  0                  2             14
Path-CBRP-MTZ-Prep            13                  0             19
Path-CBRP-Prep                 1                  0              0
[1] "Pairwise win counts for LB (Lower is better) [with '-' on diagonal]:"
                   Path-CBRP-MTZ Path-CBRP-MTZ-Prep Path-CBRP-Prep
Path-CBRP-MTZ      "-"           "2"                "14"          
Path-CBRP-MTZ-Prep "13"          "-"                "19"          
Path-CBRP-Prep     "1"           "0"                "-"           
Win counts Among Model UBs
% latex table generated in R 4.5.2 by xtable 1.8-4 package
% Thu Nov  6 21:14:50 2025
\begin{table}[ht]
\centering
\begin{tabular}{lccc}
  \hline
 & Path-CBRP-MTZ & Path-CBRP-MTZ-Prep & Path-CBRP-Prep \\ 
  \hline
Path-CBRP-MTZ & - & 2 & 14 \\ 
  Path-CBRP-MTZ-Prep & 13 & - & 19 \\ 
  Path-CBRP-Prep & 1 & 0 &

In [ ]:
# ============================================================================
# COMPREHENSIVE ANALYSIS FOR COMPUTATIONAL EXPERIMENTS SECTION
# ============================================================================

# Helper function to get runtime (handles different column names)
get_runtime <- function(df) {
  runtime_col <- intersect(c("Runtime (s)", "Time (s)"), colnames(df))
  if (length(runtime_col) > 0) return(df[[runtime_col[1]]])
  return(rep(NA, nrow(df)))
}

# Helper function to get attended blocks
get_attended <- function(df) {
  blocks_col <- intersect(c("Attended Blocks", "Attended"), colnames(df))
  if (length(blocks_col) > 0) return(df[[blocks_col[1]]])
  return(rep(NA, nrow(df)))
}

# ============================================================================
# COMPUTE KEY METRICS FOR EACH METHODOLOGY
# ============================================================================

compute_metrics <- function(df, method_name) {
  metrics <- list(method = method_name)
  metrics$n_instances <- nrow(df)
  
  # Optimal solutions (gap == 0)
  metrics$optimal_count <- sum(df[["gap (%)"]] == 0, na.rm = TRUE)
  metrics$optimal_pct <- round(100 * metrics$optimal_count / metrics$n_instances, 1)
  
  # Gap statistics
  metrics$avg_gap <- round(mean(df[["gap (%)"]], na.rm = TRUE), 4)
  metrics$max_gap <- round(max(df[["gap (%)"]], na.rm = TRUE), 4)
  
  # LB and UB
  metrics$avg_LB <- round(mean(df$LB, na.rm = TRUE), 2)
  metrics$avg_UB <- round(mean(df$UB, na.rm = TRUE), 2)
  
  # Runtime
  runtimes <- get_runtime(df)
  metrics$avg_runtime <- round(mean(runtimes, na.rm = TRUE), 2)
  metrics$max_runtime <- round(max(runtimes, na.rm = TRUE), 2)
  metrics$min_runtime <- round(min(runtimes, na.rm = TRUE), 4)
  
  # Attended blocks
  attended <- get_attended(df)
  metrics$avg_attended <- round(mean(attended, na.rm = TRUE), 1)
  
  # Problem size
  metrics$avg_V <- round(mean(as.numeric(df[["||V||"]]), na.rm = TRUE), 1)
  metrics$avg_A <- round(mean(as.numeric(df[["||A||"]]), na.rm = TRUE), 1)
  metrics$avg_B <- round(mean(as.numeric(df[["||B||"]]), na.rm = TRUE), 1)
  
  return(metrics)
}

# Compute metrics for all methodologies
all_metrics <- lapply(names(data_list), function(name) {
  compute_metrics(data_list[[name]], name)
})
names(all_metrics) <- names(data_list)

# ============================================================================
# DISPLAY SUMMARY TABLE
# ============================================================================

cat("\n")
cat("===============================================================================\n")
cat("SUMMARY METRICS FOR ALL METHODOLOGIES\n")
cat("===============================================================================\n\n")

# Create summary dataframe
summary_df <- data.frame(
  Method = sapply(all_metrics, function(x) x$method),
  Inst = sapply(all_metrics, function(x) x$n_instances),
  Opt = sapply(all_metrics, function(x) x$optimal_count),
  Opt_Pct = sapply(all_metrics, function(x) paste0(x$optimal_pct, "%")),
  Avg_Gap = sapply(all_metrics, function(x) paste0(x$avg_gap, "%")),
  Avg_Time = sapply(all_metrics, function(x) x$avg_runtime),
  Avg_Attend = sapply(all_metrics, function(x) x$avg_attended),
  Avg_LB = sapply(all_metrics, function(x) x$avg_LB),
  Avg_UB = sapply(all_metrics, function(x) x$avg_UB),
  stringsAsFactors = FALSE
)
rownames(summary_df) <- NULL
print(summary_df)

# ============================================================================
# PREPROCESSING IMPACT ANALYSIS
# ============================================================================

cat("\n")
cat("===============================================================================\n")
cat("PREPROCESSING IMPACT ANALYSIS\n")
cat("===============================================================================\n")

# Path-CBRP vs Path-CBRP-Prep
cat("\n--- Path-CBRP vs Path-CBRP-Prep (Exponential Formulation) ---\n")
exp <- data_list[["Path-CBRP"]]
exp_prep <- data_list[["Path-CBRP-Prep"]]

exp_V <- mean(as.numeric(exp[["||V||"]]), na.rm = TRUE)
exp_prep_V <- mean(as.numeric(exp_prep[["||V||"]]), na.rm = TRUE)
cat("Problem Size |V|:", round(exp_V, 1), "->", round(exp_prep_V, 1), 
    "(", round(100*(1-exp_prep_V/exp_V), 1), "% reduction)\n")

exp_time <- mean(get_runtime(exp), na.rm = TRUE)
exp_prep_time <- mean(get_runtime(exp_prep), na.rm = TRUE)
cat("Avg Runtime:", round(exp_time, 2), "s ->", round(exp_prep_time, 2), "s",
    "(", round(100*(1-exp_prep_time/exp_time), 1), "% reduction)\n")

exp_gap <- mean(exp[["gap (%)"]], na.rm = TRUE)
exp_prep_gap <- mean(exp_prep[["gap (%)"]], na.rm = TRUE)
cat("Avg Gap:", round(exp_gap, 4), "% ->", round(exp_prep_gap, 4), "%\n")

# Path-CBRP-MTZ vs Path-CBRP-MTZ-Prep
cat("\n--- Path-CBRP-MTZ vs Path-CBRP-MTZ-Prep (MTZ Formulation) ---\n")
mtz <- data_list[["Path-CBRP-MTZ"]]
mtz_prep <- data_list[["Path-CBRP-MTZ-Prep"]]

mtz_time <- mean(get_runtime(mtz), na.rm = TRUE)
mtz_prep_time <- mean(get_runtime(mtz_prep), na.rm = TRUE)
cat("Avg Runtime:", round(mtz_time, 2), "s ->", round(mtz_prep_time, 2), "s",
    "(", round(100*(1-mtz_prep_time/mtz_time), 1), "% reduction)\n")

mtz_gap <- mean(mtz[["gap (%)"]], na.rm = TRUE)
mtz_prep_gap <- mean(mtz_prep[["gap (%)"]], na.rm = TRUE)
cat("Avg Gap:", round(mtz_gap, 4), "% ->", round(mtz_prep_gap, 4), "%\n")

# Path-CBRP-Frac vs Path-CBRP-Frac-Prep
cat("\n--- Path-CBRP-Frac vs Path-CBRP-Frac-Prep (Fractional Cuts) ---\n")
frac <- data_list[["Path-CBRP-Frac"]]
frac_prep <- data_list[["Path-CBRP-Frac-Prep"]]

frac_gap <- mean(frac[["gap (%)"]], na.rm = TRUE)
frac_prep_gap <- mean(frac_prep[["gap (%)"]], na.rm = TRUE)
cat("Avg Gap:", round(frac_gap, 4), "% ->", round(frac_prep_gap, 4), "% (WORSENED)\n")

frac_opt <- sum(frac[["gap (%)"]] == 0, na.rm = TRUE)
frac_prep_opt <- sum(frac_prep[["gap (%)"]] == 0, na.rm = TRUE)
cat("Optimal solutions:", frac_opt, "->", frac_prep_opt, "\n")

# ============================================================================
# GENERATE LATEX TABLE FOR PAPER
# ============================================================================

cat("\n")
cat("===============================================================================\n")
cat("LATEX TABLE - COMPUTATIONAL RESULTS\n")
cat("===============================================================================\n\n")

library(xtable)
latex_df <- summary_df
colnames(latex_df) <- c("Method", "Inst.", "Opt.", "Opt.(%)", "Avg.Gap(%)", 
                         "Avg.Time(s)", "Avg.Blocks", "Avg.LB", "Avg.UB")
print(xtable(latex_df, 
             caption = "Summary of computational results for all methodologies.", 
             label = "tab:summary_results",
             align = c("l", "l", rep("c", 8))),
      include.rownames = FALSE,
      sanitize.text.function = identity)